# S3 buckets

Creates two S3 buckets: the **data lake bucket** and the **Athena results bucket**.
Run the cells top to bottom. Each cell is safe to run twice, and prints the AWS CLI command it runs (`$ aws ...`).
Names and region come from `config/settings.json`.

## 1. Setup

In [1]:
import sys
from pathlib import Path

# Works whether Jupyter starts in the project root or in notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "infra"))

# aws() runs the AWS CLI and prints the command and its output (see infra/aws_helpers.py)
from aws_helpers import aws, aws_json, whoami, REGION, LAKE, RESULTS

print("identity:", whoami())
print("region  :", REGION)
print("lake    :", LAKE)
print("results :", RESULTS)
assert REGION == "us-east-1", "this project is built for us-east-1"

identity: user/Ahmed
region  : us-east-1
lake    : patientflow-datalake-zhmkbf
results : patientflow-athena-query-results-zhmkbf


## 2. Bucket setup function
Each bucket gets: **create** (only if missing), **block all public access**, **default encryption** (SSE-S3).

In `us-east-1`, `create-bucket` takes no location setting; every other region needs `LocationConstraint`.

In [2]:
def ensure_bucket(name):
    """Create the bucket if missing, then apply public-access block and default encryption."""
    exists = aws("s3api", "head-bucket", "--bucket", name, check=False).returncode == 0
    if exists:
        print(f"bucket {name} already exists - skipping create\n")
    else:
        aws("s3api", "create-bucket", "--bucket", name, "--region", REGION)

    aws("s3api", "put-public-access-block", "--bucket", name,
        "--public-access-block-configuration",
        "BlockPublicAcls=true,IgnorePublicAcls=true,BlockPublicPolicy=true,RestrictPublicBuckets=true")

    aws("s3api", "put-bucket-encryption", "--bucket", name,
        "--server-side-encryption-configuration", "file://infra/s3/encryption.json")

## 3. Data lake bucket

In [3]:
ensure_bucket(LAKE)

$ aws s3api head-bucket --bucket patientflow-datalake-zhmkbf


{
    "BucketArn": "arn:aws:s3:::patientflow-datalake-zhmkbf",
    "BucketRegion": "us-east-1",
    "AccessPointAlias": false
}
bucket patientflow-datalake-zhmkbf already exists - skipping create

$ aws s3api put-public-access-block --bucket patientflow-datalake-zhmkbf --public-access-block-configuration BlockPublicAcls=true,IgnorePublicAcls=true,BlockPublicPolicy=true,RestrictPublicBuckets=true


$ aws s3api put-bucket-encryption --bucket patientflow-datalake-zhmkbf --server-side-encryption-configuration file://infra/s3/encryption.json


## 4. Athena results bucket

In [4]:
ensure_bucket(RESULTS)

$ aws s3api head-bucket --bucket patientflow-athena-query-results-zhmkbf


{
    "BucketArn": "arn:aws:s3:::patientflow-athena-query-results-zhmkbf",
    "BucketRegion": "us-east-1",
    "AccessPointAlias": false
}
bucket patientflow-athena-query-results-zhmkbf already exists - skipping create

$ aws s3api put-public-access-block --bucket patientflow-athena-query-results-zhmkbf --public-access-block-configuration BlockPublicAcls=true,IgnorePublicAcls=true,BlockPublicPolicy=true,RestrictPublicBuckets=true


$ aws s3api put-bucket-encryption --bucket patientflow-athena-query-results-zhmkbf --server-side-encryption-configuration file://infra/s3/encryption.json


## 5. Expire old query results after 7 days
Rule defined in `infra/s3/lifecycle_athena_results.json`.

In [5]:
aws("s3api", "put-bucket-lifecycle-configuration", "--bucket", RESULTS,
    "--lifecycle-configuration", "file://infra/s3/lifecycle_athena_results.json")

$ aws s3api put-bucket-lifecycle-configuration --bucket patientflow-athena-query-results-zhmkbf --lifecycle-configuration file://infra/s3/lifecycle_athena_results.json


{
    "TransitionDefaultMinimumObjectSize": "all_storage_classes_128K"
}


<aws exit code 0>

## 6. Verify
Read the settings back from AWS; an `assert` fails if anything differs from the design.

In [6]:
for bucket in (LAKE, RESULTS):
    pab = aws_json("s3api", "get-public-access-block", "--bucket", bucket)["PublicAccessBlockConfiguration"]
    assert all(pab.values()), f"{bucket}: public access is not fully blocked"

    enc = aws_json("s3api", "get-bucket-encryption", "--bucket", bucket)
    algo = enc["ServerSideEncryptionConfiguration"]["Rules"][0]["ApplyServerSideEncryptionByDefault"]["SSEAlgorithm"]
    assert algo == "AES256", f"{bucket}: unexpected encryption {algo}"
    print(f"OK  {bucket}: public access blocked, encryption {algo}")

rule = aws_json("s3api", "get-bucket-lifecycle-configuration", "--bucket", RESULTS)["Rules"][0]
assert rule["Expiration"]["Days"] == 7 and rule["Status"] == "Enabled"
print(f"OK  {RESULTS}: results expire after {rule['Expiration']['Days']} days")

OK  patientflow-datalake-zhmkbf: public access blocked, encryption AES256


OK  patientflow-athena-query-results-zhmkbf: public access blocked, encryption AES256


OK  patientflow-athena-query-results-zhmkbf: results expire after 7 days


## What these buckets are for

| Bucket | Used for | Used in |
|---|---|---|
| Data lake | One bucket for every layer, under folders: `landing/`, `bronze/patient_records/`, `silver/`, `gold/`, `glue_scripts/`, `data_quality_results/`. Folders appear when files are written; nothing to create. | Phases 3d - 8 |
| Athena results | Every Athena query writes its result file here; files expire after 7 days. | Phase 3 (Athena) onwards |

**Next:** `create_catalog_databases` creates the Glue databases and the Athena workgroup.